In [1]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

## Consistency

In [2]:
render_code("fence.c", show=["#ifndef MFENCE","endif"])

// fence.c:13-28 (16 lines)
#ifndef MFENCE
void* modifya(void *z)
{
  while(!go);
  a=1;
  x=b;
  return NULL;
}
void* modifyb(void *z)
{
  while(!go);
  b=1;
  y=a;
  return NULL;
}
#endif

In [3]:
! make clean; make fence

rm -f testloop testloop_O3 coherence blockmm blockmm_pthread value_of_i vadd_sse fence mfence vadd_sse_two_threads vadd_sse_two_threads_chunk vadd_mt vadd_mt_chunk testloop_volatile testloop_O3_volatile coherence_lock
rm -rf *.dSYM
cc -O3 -Wno-format-zero-length -Wno-implicit-function-declaration fence.c -o fence -lpthread


In [4]:
!rm fence.txt;  touch fence.txt
!lscpu | grep "Model name:"
!cd ~/courses/CSE142/demo/multiprocessor; for i in {1..5000}; do ./fence 2>> fence.txt; done; for i in {1..5000}; do ./fence 2>> fence.txt; done;

Model name:                           13th Gen Intel(R) Core(TM) i7-13700F


In [5]:
! grep "(0, 0)" fence.txt |wc
! grep "(0, 1)" fence.txt |wc
! grep "(1, 1)" fence.txt |wc
! grep "(1, 0)" fence.txt |wc

      2       8      38
   8558   34232  162602
      0       0       0
   1440    5760   27360


In [6]:
! rm fence_amd.txt; touch fence_amd.txt
! ssh htseng@blissey "lscpu | grep 'Model name:'; cd ~/courses/CSE142/demo/multiprocessor; for i in {1..10000}; do ./fence 2>> fence_amd.txt; done;"
! grep "(0, 0)" fence_amd.txt |wc
! grep "(0, 1)" fence_amd.txt |wc
! grep "(1, 1)" fence_amd.txt |wc
! grep "(1, 0)" fence_amd.txt |wc

Model name:                           AMD Ryzen 7 5700X 8-Core Processor
      9      36     171
   9984   39936  189696
      0       0       0
      7      28     133


### "mfence" instructions:

An instruction that forces all updates must finish before making progress after this instruction.

In [26]:
render_code("fence.c", show=["#ifdef MFENCE", "#endif"])

// fence.c:41-59 (19 lines)
#ifdef MFENCE
#define _update_var(_v, _x) { _v = (_x); asm("mfence"); }
void* modifya(void *z)
{
  a=1;
  asm volatile("mfence" ::: "memory");
//  _update_var(a,1);
  x=b;
  return NULL;
}
void* modifyb(void *z)
{
  b=1;
  asm volatile("mfence" ::: "memory");
//  _update_var(b,1);
  y=a;
  return NULL;
}
#endif

In [23]:
!make clean mfence
!rm mfence.txt; touch mfence.txt
!for i in {1..10000}; do ./mfence 2>> mfence.txt; done;

rm -f testloop testloop_O3 coherence blockmm blockmm_pthread value_of_i vadd_sse fence mfence vadd_sse_two_threads vadd_sse_two_threads_chunk vadd_mt vadd_mt_chunk testloop_volatile testloop_O3_volatile coherence_lock
rm -rf *.dSYM
cc -DMFENCE -O3 -Wno-format-zero-length -Wno-implicit-function-declaration fence.c -o mfence -lpthread


In [24]:
! grep "(0, 0)" mfence.txt |wc
! grep "(0, 1)" mfence.txt |wc
! grep "(1, 1)" mfence.txt |wc
! grep "(1, 0)" mfence.txt |wc

      0       0       0
   8459   33836  160721
     12      48     228
   1529    6116   29051


In [25]:
! rm mfence_amd.txt; touch mfence_amd.txt
! ssh htseng@blissey "cd ~/courses/CSE142/demo/multiprocessor; for i in {1..10000}; do ./mfence 2>> mfence_amd.txt; done;"
! grep "(0, 0)" mfence_amd.txt |wc
! grep "(0, 1)" mfence_amd.txt |wc
! grep "(1, 1)" mfence_amd.txt |wc
! grep "(1, 0)" mfence_amd.txt |wc

      0       0       0
   9998   39992  189962
      0       0       0
      2       8      38
